# ricetein smoke test (Kaggle plumbing)

**Pattern for every Kaggle notebook in this project.** Kaggle loads some packages (e.g. numpy) into
this notebook's own Python before our first cell runs, so our locked versions can't be installed here
safely. Instead, `setup_env()` builds a separate environment with the exact locked packages, and
`run()` runs our code (`python -m src....`) in a fresh process inside it. Results go to
`/kaggle/working/runs/` as usual.

The check covers: packages and where they load from, GPU, both datasets, a run record, and DINOv3
(needs the `HF_TOKEN` secret attached to this notebook; see `docs/KAGGLE_WORKFLOW.md` §9).

Run: `./scripts/kaggle_push.sh 00-smoke`, then `./scripts/kaggle_pull.sh 00-smoke`.

In [ ]:
# 1. Make our code importable here (standard library only), then build the isolated environment.
import sys
from pathlib import Path

_root = Path("/kaggle/input")
CODE_DIR = next(p for p in [_root / "ricetein-code", *sorted(_root.glob("*/*/ricetein-code"))] if p.is_dir())
sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR, "| version:", (CODE_DIR / "CODE_VERSION").read_text().strip())

from src.kaggle_bootstrap import run, setup_env
setup_env()

In [ ]:
# 2. Secrets are read here (Kaggle's secrets helper only exists in this process) and passed in.
from kaggle_secrets import UserSecretsClient

try:
    SECRETS = {"HF_TOKEN": UserSecretsClient().get_secret("HF_TOKEN")}
except Exception as e:
    raise RuntimeError("Attach the HF_TOKEN secret to this notebook (Add-ons -> Secrets).") from e

In [ ]:
# 3. Run the check in the isolated environment.
run("src.smoke_check", "--require-dinov3", env=SECRETS)